## FX Multi Cross Products
The below notebook shows you how to model various multi-cross FX products, dual binary, worst of, contingent ko etc
In general: Each currency pair is modelled as a sub-instrument set as an array onto the `legs` property.

In [ ]:
from gs_quant.session import GsSession, Environment
from gs_quant import risk
from gs_quant.common import OptionType

In [ ]:
# external users should substitute their client id and secret
GsSession.use(Environment.PROD, client_id=None, client_secret=None, scopes=('run_analytics',))

Note: At time of writing, 20May26, USDJPY is ~159 and EURUSD is ~1.16. Adjust example as necessary if spot has moved.

### Dual Binary
Dual (or tri/quad) binaries are modelled using the FXMultiCrossBinary calculator
Each currency pair is modelled as a sub leg set as an array onto the `legs` property  
e.g. To model a `3m eurusd<1.19  usdjpy>160 dual binary`

In [ ]:
from gs_quant.instrument import FXMultiCrossBinary, FXMultiCrossBinaryLeg

dual_binary = FXMultiCrossBinary(
    legs=(
        FXMultiCrossBinaryLeg(pair='EURUSD', option_type=OptionType.Binary_Put, strike_price=1.19),
        FXMultiCrossBinaryLeg(pair='USDJPY', option_type=OptionType.Binary_Call, strike_price=160),
    ),
    premium=0,
)
print(dual_binary.calc(risk.FairPremiumInPercent) * 100)

### Dual Double Binary
"Double" binaries are for modelling a payout of `X<spot_ata_expiry<Y` they are modelled very similar to dual binary but instead of an `option_type` and `strike_price` three is an `upper_strike_price` and `lower_strike_price`
e.g. to model a fixed payout when `3m 1.15<eurusd<1.19 158<usdjpy>160 dual`

In [ ]:
from gs_quant.instrument import FXMultiCrossDoubleBinary, FXMultiCrossDoubleBinaryLeg

dual_double_binary = FXMultiCrossDoubleBinary(
    legs=(
        FXMultiCrossDoubleBinaryLeg(pair='EURUSD', lower_barrier_level=1.15, upper_barrier_level=1.19),
        FXMultiCrossDoubleBinaryLeg(pair='USDJPY', lower_barrier_level=158, upper_barrier_level=160),
    ),
    premium=0,
)
print(dual_double_binary.calc(risk.FairPremiumInPercent) * 100)

### Contingent KO (Dual Double Knockout)
To model an option in one currency (with or without a knockout) with a knockout in a second currency pair you can use `FXDualDoubleKnockout`  
The **first** leg corresponds to the option payout, the **second** leg is just for the additional knockout.  
The option_type can also be `Binary_Put` or `Binary_Call`  
**Only** two `legs` are supported.  
e.g. `3mnth eurusd 1.19 put with usdjpy 158 ko` (assumed up-and-out-ko based on USDJPY spot at time of writing)

In [ ]:
from gs_quant.instrument import FXDualDoubleKnockout, FXDualDoubleKnockoutLeg

dual_ko = FXDualDoubleKnockout(
    expiration_date="3m",
    option_type=OptionType.Put,
    strike_price=1.19,
    legs=(FXDualDoubleKnockoutLeg(pair='EURUSD'), FXDualDoubleKnockoutLeg(pair='USDJPY', upper_barrier_level=165)),
    premium=0,
)
print(dual_ko.calc(risk.FairPremiumInPercent) * 100)

### Dual DNT
The above model can also be used to make a dual DNT (double no touch) by setting payoff to a binary call at 0 and setting the barriers of both crosses.  
e.g. `2mnth EURUSD 1.15/1.20 USDJPY 150/160 dual DNT`

In [ ]:
from gs_quant.instrument import FXDualDoubleKnockout, FXDualDoubleKnockoutLeg

dual_dnt = FXDualDoubleKnockout(
    expiration_date="2m",
    option_type=OptionType.Binary_Call,
    strike_price=0,
    legs=(
        FXDualDoubleKnockoutLeg(pair='EURUSD', lower_barrier_level=1.15, upper_barrier_level=1.20),
        FXDualDoubleKnockoutLeg(pair='USDJPY', lower_barrier_level=150, upper_barrier_level=165),
    ),
    premium=0,
)
print(dual_dnt.calc(risk.FairPremiumInPercent) * 100)

### Worst of
To model a worst of option i.e. the minimium payout of two options. You use the FXWorstOf instrument, there must be a common currency across all the crosses (e.g. USD in the below example).  
`worst of 3mnth EURUSD put 1.19, USDJPY call 158`

In [ ]:
from gs_quant.instrument import FXWorstOf, FXWorstOfLeg

worst_of = FXWorstOf(
    legs=(
        FXWorstOfLeg(expiration_date="3m", pair='EURUSD', option_type=OptionType.Put, strike_price=1.19),
        FXWorstOfLeg(expiration_date="3m", pair='USDJPY', option_type=OptionType.Call, strike_price=165),
    ),
    premium=0,
)
print(worst_of.calc(risk.FairPremiumInPercent) * 100)

### Worst of with KO
To model a worst of option with a knockout (american barrier) on at least one cross you can use the FXWorstOfKO instrument. Knock up or down will default based on current spot. Don't set the `barrier_level` or set it to `-1` to have no barrier for that cross.  
e.g. `3mnth worst of EURUSD put 1.19 rko 1.15, USDJPY call 158`

In [ ]:
from gs_quant.instrument import FXWorstOfKO, FXWorstOfKOLeg

worst_of_ko = FXWorstOfKO(
    legs=(
        FXWorstOfKOLeg(
            expiration_date="3m", pair='EURUSD', option_type=OptionType.Put, strike_price=1.19, barrier_level=1.15
        ),
        FXWorstOfKOLeg(expiration_date="3m", pair='USDJPY', option_type=OptionType.Call, strike_price=165),
    )
)
print(worst_of_ko.calc(risk.FairPremiumInPercent) * 100)

### FXCorrelationSwap
A correlation swap, the strike will default to the fair correlation strike.
e.g.  `1mnth EURUSD USDJPY corr swap`

In [ ]:
from gs_quant.instrument import FXCorrelationSwap, FXCorrelationSwapLeg

corr_swap = FXCorrelationSwap(
    last_fixing_date="1m", legs=(FXCorrelationSwapLeg(pair='EURUSD'), FXCorrelationSwapLeg(pair='USDJPY'))
)
print(corr_swap.calc(risk.FairPremiumInPercent) * 100)

## Risk measures
You will see that various risk measures that would normally return a single number now return a dataframe, indexed by the leg/cross

In [ ]:
worst_of_ko.calc(risk.FXSpot)

In [ ]:
worst_of_ko.calc(risk.FXDeltaHedge)